# Afternoon class 30/08 — Worksheet 14 SOLUTIONS: Excel and capstone   (L03)

Every cell below was executed in the lab image (pandas 3.0.5) against the real
`data/quarterly.xlsx`, and the quoted output is what it actually printed.

Question 5 is the one to re-read. The default `to_csv` adds a column to your
file that was never in your data, and it comes back on the next read.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 14 — Excel, writing, capstone. Run this once.
import pandas as pd

# data/quarterly.xlsx holds the same 300 orders split into four sheets by
# the quarter of OrderDate.
sales = pd.read_csv("data/sales.csv", parse_dates=["OrderDate"])

print("sales.csv:", sales.shape)
print("date range:", sales["OrderDate"].min().date(), "to", sales["OrderDate"].max().date())

PART A — reading a workbook

### Question 1

`sheet_name="Q1"` -> a `DataFrame`, `(65, 10)`.

One named sheet, one DataFrame. `OrderDate` came back as a real timestamp
without `parse_dates` — Excel stores dates as a typed cell, so unlike CSV
there is nothing to infer.

In [ ]:
q1 = pd.read_excel("data/quarterly.xlsx", sheet_name="Q1")
print("type: ", type(q1).__name__)
print("shape:", q1.shape)
print()
print(q1.head(3)[["OrderID", "OrderDate", "Region", "Sales"]])

### Question 2

`sheet_name=None` -> a **`dict`** with keys `['Q1', 'Q2', 'Q3', 'Q4']`. -> `(65, 10)`, `(86, 10)`, `(65, 10)`, `(84, 10)`, totalling **300**.

`None` means 'all sheets', not 'no sheet', and it changes the return type
from DataFrame to dict-of-DataFrames. Code that does `.shape` on the result
breaks the moment someone switches to `None`.

The four sheets total exactly the 300 rows of `sales.csv`, which is the
check worth doing whenever data arrives split across tabs: sum the parts
and compare with what you expected. Quarterly workbooks are where rows go
missing, because nobody looks at all four tabs at once.

In [ ]:
book = pd.read_excel("data/quarterly.xlsx", sheet_name=None)
print("type:", type(book).__name__)
print("keys:", list(book.keys()))
print()
total = 0
for name, df in book.items():
    print("%-4s -> %s" % (name, df.shape))
    total += len(df)
print()
print("rows across sheets:", total, "| sales.csv:", len(sales))

### Question 3

`sheet_name=0` -> `(65, 10)`, `.equals()` the `"Q1"` sheet is `True`.

Identical, today. `0` means 'whichever sheet is leftmost', and tab order
is a presentation detail that people rearrange freely — dragging Q4 to the
front to make it easier to find changes no data and silently changes which
quarter your report describes.

Names are stable and self-documenting; positions are neither. The same
argument as `.loc` over `.iloc` in worksheet 04.

In [ ]:
by_name = pd.read_excel("data/quarterly.xlsx", sheet_name="Q1")
by_pos = pd.read_excel("data/quarterly.xlsx", sheet_name=0)
print("shape:", by_pos.shape)
print("same as 'Q1':", by_pos.equals(by_name))

# 0 means "whichever sheet is first". Someone reordering the tabs in Excel
# -- a thing people do for readability, without touching any data -- silently
# changes which quarter your report is about.

### Question 4

Combined `(300, 10)`. -> from sheets `236825.00350000002`, from CSV `236825.0035`. **`equal: False`**.

The same 300 orders, the same column, and the totals differ in the last
two digits — the second independent instance of worksheet 12's finding.

Here no chunking is involved at all. The rows simply arrived in a different
order: grouped by quarter across four sheets, rather than in the CSV's
original sequence. That reordering changes the grouping of the additions,
which changes the rounding, which changes the last bits.

So the rule generalises beyond `chunksize`: **any** operation that changes
the order in which floats are added can change the total. Splitting a file,
concatenating it back, sorting, grouping, reading from a different format —
all of them. Reconciling a workbook against its source CSV with `==` will
fail for reasons that have nothing to do with the data.

In [ ]:
book = pd.read_excel("data/quarterly.xlsx", sheet_name=None)
combined = pd.concat(book.values(), ignore_index=True)
print("combined:", combined.shape)
print()
print("from sheets:", repr(combined["Sales"].sum()))
print("from csv:   ", repr(sales["Sales"].sum()))
print("equal:", combined["Sales"].sum() == sales["Sales"].sum())

PART B — writing, and the column you did not ask for

### Question 5

Default first line `',OrderID,Sales\n'` — it starts with a comma. `index=False` -> `'OrderID,Sales\n'`. -> reading the default back gives columns `['Unnamed: 0', 'OrderID', 'Sales']`.

The default `to_csv` writes the index as an unnamed first column. On the
way back in, Pandas has an anonymous column to name and calls it
`Unnamed: 0`.

So a save-then-load cycle *added a column to your data*. Do it twice and
you get `Unnamed: 0` and `Unnamed: 0.1`. Every analyst has seen a file with
three of them, and each one is a generation of someone forgetting
`index=False`.

The rule is simple: pass `index=False` unless the index holds real
information. Q9 is the case where it does.

In [ ]:
small = sales.head(3)[["OrderID", "Sales"]]
small.to_csv("/tmp/with_index.csv")
small.to_csv("/tmp/no_index.csv", index=False)

print("default    first line:", repr(open("/tmp/with_index.csv").readline()))
print("index=False first line:", repr(open("/tmp/no_index.csv").readline()))
print()
back = pd.read_csv("/tmp/with_index.csv")
print("reading the default file back -> columns:", list(back.columns))
print(back)

### Question 6

`Large -> (59, 10)`, `Small -> (241, 10)`, totalling `300`.

`ExcelWriter` as a context manager keeps one workbook open while several
frames are written into it, then closes and saves on exit. Writing each
frame with a separate `to_excel` call to the same filename would overwrite
the file each time and leave you with only the last sheet.

The 59/241 split matches worksheet 07 Q1 and worksheet 09 Q7 exactly —
three different routes to the same threshold, same answer.

In [ ]:
large = sales[sales["Sales"] > 1000]
small = sales[sales["Sales"] <= 1000]

with pd.ExcelWriter("/tmp/split.xlsx") as writer:
    large.to_excel(writer, sheet_name="Large", index=False)
    small.to_excel(writer, sheet_name="Small", index=False)

back = pd.read_excel("/tmp/split.xlsx", sheet_name=None)
for name, df in back.items():
    print("%-6s -> %s" % (name, df.shape))
print()
print("total rows:", sum(len(d) for d in back.values()))

PART C — capstone: the whole pipeline

### Question 7

`(300, 10)` with `OrderDate` as `datetime64[us]`. -> missing: `Region 28`, `Discount 43`, `Profit 24`. -> **215 complete rows of 300**.

Load and check, before anything else. The dtypes confirm the numeric
columns are numeric — which, as worksheet 11 showed, is not automatic when
placeholders are present — and `isna().sum()` names the damage.

The number that matters is **215 of 300**. Only 72% of rows are complete,
and `dropna()` is one keystroke away from throwing the other 28% out.
Whether that is acceptable is a decision about the analysis, not a
formatting step, and you cannot make it without seeing this number.

In [ ]:
raw = pd.read_csv(
    "data/sales_messy.csv",
    na_values=["?", "Missing"],
    parse_dates=["OrderDate"],
)
print("shape:", raw.shape)
print()
print(raw.dtypes)
print()
print("missing per column:")
print(raw.isna().sum()[lambda s: s > 0])
print()
complete = raw.dropna()
print("complete rows:", len(complete), "of", len(raw))

### Question 8

`groupby(dropna=False)` counts **300** rows; the default counts **272**. -> the `NaN` region holds `7946.7365`. -> group total `236825.0035` **equals** the ungrouped total.

The default `groupby` silently discarded the 28 rows whose region is
missing — nearly £8,000 of sales that appear in no group and in no total
derived from those groups. The report would balance internally and be short
by 3%.

`dropna=False` gives them a `NaN` group. It is not pretty in a presentation
and it is honest: those orders exist and belong somewhere unknown.

And note the sums **do** reconcile exactly here, unlike Q4. Same data, same
float arithmetic, and this grouping happened to land on the identical
value. That is the honest summary of the whole float question: the
difference is unpredictable, sometimes zero, and never something to assert
with `==`.

In [ ]:
raw = pd.read_csv("data/sales_messy.csv", na_values=["?", "Missing"],
                  parse_dates=["OrderDate"])

with_na = raw.groupby("Region", dropna=False)["Sales"].sum()
without_na = raw.groupby("Region")["Sales"].sum()

print("groupby(dropna=False):")
print(with_na.to_string())
print()
print("rows counted with dropna=False:", raw.groupby("Region", dropna=False).size().sum())
print("rows counted with the default: ", raw.groupby("Region").size().sum())
print()
print("group total:", repr(with_na.sum()))
print("ungrouped:  ", repr(raw["Sales"].sum()))
print("equal:", with_na.sum() == raw["Sales"].sum())

### Question 9

`region_totals.csv` keeps the region names as its first column; the missing-region row appears with an **empty key**. -> workbook sheets `Clean (215, 10)` and `Summary (9, 2)`.

This is the case where the index *is* data — it holds the region names —
so writing it is right and `index=False` would have destroyed the file,
leaving a column of numbers with nothing to identify them.

That is the whole rule, and it is a judgement rather than a habit: write
the index when it carries meaning, suppress it when it is just row numbers.

Two details worth seeing in the output. The `NaN` region wrote as an empty
field — `,7946.74` — so a downstream reader gets a blank category rather
than anything labelled 'unknown'; name it explicitly with `fillna("Unknown")`
before export if a human will read it. And Quebec's `6885.9965` rounded to
`6886.0`, printing one decimal place rather than two, because `round()`
produces a number and the trailing zero was never stored.

In [ ]:
raw = pd.read_csv("data/sales_messy.csv", na_values=["?", "Missing"],
                  parse_dates=["OrderDate"])
totals = raw.groupby("Region", dropna=False)["Sales"].sum().round(2)

# Here the index IS the data -- it holds the region names -- so keep it.
totals.to_csv("/tmp/region_totals.csv")
print(open("/tmp/region_totals.csv").read())

with pd.ExcelWriter("/tmp/report.xlsx") as writer:
    raw.dropna().to_excel(writer, sheet_name="Clean", index=False)
    totals.to_frame("TotalSales").to_excel(writer, sheet_name="Summary")

back = pd.read_excel("/tmp/report.xlsx", sheet_name=None)
for name, df in back.items():
    print("%-8s -> %s" % (name, df.shape))

### Question 10

`sheet_name="Q5"` -> **raises** `ValueError: Worksheet named 'Q5' not found`.

A clear message naming exactly what it looked for. Compare with
`sheet_name=4`, which would raise a less obvious `IndexError` — another
reason to address sheets by name.

A fitting end to the day, because it is the pattern the whole class keeps
returning to: **Pandas is loud about structure and quiet about meaning.**
A sheet that does not exist stops you instantly. A sheet that exists and
contains the wrong quarter, a `?` that silently turns a numeric column into
text, a `groupby` that drops 28 rows, a total that disagrees with itself in
the twelfth decimal — none of those raise anything at all.

The errors are not the hard part. The things that run are.

In [ ]:
print(pd.read_excel("data/quarterly.xlsx", sheet_name="Q5"))